# 1- Apriori:

In [14]:
# Apriori implementation using mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder # converts a list of transactions (list of item strings) into a binary matrix (one-hot encoding).
from mlxtend.frequent_patterns import apriori, association_rules

# Dataset
transactions = [
    ['E','K','M','N','O','Y'],        # T1
    ['D','E','K','N','O','Y'],        # T2
    ['A','E','K','M'],                # T3
    ['C','K','M','U','Y'],            # T4
    ['C','E','I','K','O']             # T5
]

# One-hot encoding
'''
|    | A | C | D | E | I | K | M | N | O | U | Y |
| -- | - | - | - | - | - | - | - | - | - | - | - |
| T1 | 0 | 0 | 0 | 1 | 0 | 1 | 1 | 1 | 1 | 0 | 1 |
| T2 | 0 | 0 | 1 | 1 | 0 | 1 | 0 | 1 | 1 | 0 | 1 |
| T3 | 1 | 0 | 0 | 1 | 0 | 1 | 1 | 0 | 0 | 0 | 0 |
| T4 | 0 | 1 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 1 | 1 |
| T5 | 0 | 1 | 0 | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 0 |

'''
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
te_array

array([[False, False, False,  True, False,  True,  True,  True,  True,
        False,  True],
       [False, False,  True,  True, False,  True, False,  True,  True,
        False,  True],
       [ True, False, False,  True, False,  True,  True, False, False,
        False, False],
       [False,  True, False, False, False,  True,  True, False, False,
         True,  True],
       [False,  True, False,  True,  True,  True, False, False,  True,
        False, False]])

In [15]:

df = pd.DataFrame(te_array, columns=te.columns_)
df

,A,C,D,E,I,K,M,N,O,U,Y
0,False,False,False,True,False,True,True,True,True,False,True
1,False,False,True,True,False,True,False,True,True,False,True
2,True,False,False,True,False,True,True,False,False,False,False
3,False,True,False,False,False,True,True,False,False,True,True
4,False,True,False,True,True,True,False,False,True,False,False


In [16]:
# Apriori (support as fraction)
frequent_itemsets = apriori(
    df,
    min_support=0.60,   # support count = 3
    use_colnames=True
)

# Generate association rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.80
)

print("Frequent Itemsets:")
print(frequent_itemsets)

print("\nAssociation Rules:")
print(rules[['antecedents','consequents','support','confidence','lift']])

Frequent Itemsets:
    support   itemsets
0       0.8        (E)
1       1.0        (K)
2       0.6        (M)
3       0.6        (O)
4       0.6        (Y)
5       0.8     (K, E)
6       0.6     (E, O)
7       0.6     (K, M)
8       0.6     (K, O)
9       0.6     (K, Y)
10      0.6  (E, K, O)

Association Rules:
  antecedents consequents  support  confidence  lift
0         (K)         (E)      0.8         0.8  1.00
1         (E)         (K)      0.8         1.0  1.00
2         (O)         (E)      0.6         1.0  1.25
3         (M)         (K)      0.6         1.0  1.00
4         (O)         (K)      0.6         1.0  1.00
5         (Y)         (K)      0.6         1.0  1.00
6      (O, E)         (K)      0.6         1.0  1.00
7      (K, O)         (E)      0.6         1.0  1.25
8         (O)      (K, E)      0.6         1.0  1.25


c:\Users\yomna\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


We generate lots of useless combinations
(candidates) that are pruned later.

To overcome this limitation, use FP Growth
algorithm.

# 2- FP-Growth

In [ ]:
# FP-Growth / FP-Tree using mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Transactions from the image
# Note: T5 has O repeated in the image -> in market-basket mining we count it once per transaction.
transactions = [
    ['E','K','M','N','O','Y'],   # T1
    ['D','E','K','N','O','Y'],   # T2
    ['A','E','K','M'],           # T3
    ['C','K','M','U','Y'],       # T4
    ['C','E','I','K','O']        # T5
]

# One-hot encode transactions
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

# Given: min support count = 3
# We have 5 transactions => min_support fraction = 3/5 = 0.6
min_support = 3 / len(transactions)  # 0.6

# FP-Growth to get frequent itemsets
freq_itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)
freq_itemsets = freq_itemsets.sort_values(["support", "itemsets"], ascending=[False, True]).reset_index(drop=True)

print("=== Frequent Itemsets (FP-Growth) ===")
print(freq_itemsets)

# OPTIONAL: generate association rules
# Set min_confidence as you like (e.g., 0.5)
min_conf = 0.8
rules = association_rules(freq_itemsets, metric="confidence", min_threshold=min_conf)
rules = rules.sort_values(["confidence", "support", "lift"], ascending=[False, False, False]).reset_index(drop=True)

print(f"\n=== Association Rules (confidence >= {min_conf}) ===")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]])


=== Frequent Itemsets (FP-Growth) ===
    support   itemsets
0       1.0        (K)
1       0.8        (E)
2       0.8     (K, E)
3       0.6        (Y)
4       0.6        (O)
5       0.6        (M)
6       0.6     (K, Y)
7       0.6     (E, O)
8       0.6     (K, O)
9       0.6  (E, K, O)
10      0.6     (K, M)

=== Association Rules (confidence >= 0.8) ===
  antecedents consequents  support  confidence  lift
0         (E)         (K)      0.8         1.0  1.00
1         (O)         (E)      0.6         1.0  1.25
2      (K, O)         (E)      0.6         1.0  1.25
3         (O)      (K, E)      0.6         1.0  1.25
4         (Y)         (K)      0.6         1.0  1.00
5         (O)         (K)      0.6         1.0  1.00
6      (O, E)         (K)      0.6         1.0  1.00
7         (M)         (K)      0.6         1.0  1.00
8         (K)         (E)      0.8         0.8  1.00


c:\Users\yomna\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)
